# Incidence dataset preprocessing

## Product classification via LLM

First we use the full category list from training and testing sets to generate a set of broad categories and product flags to be used for regression

In [154]:
from openai import OpenAI
from pydantic import BaseModel, Field
import json
import pandas as pd
from numpy import log
import numpy as np
from collections import defaultdict



from typing import Literal


In [3]:
train_df = pd.read_csv("incidence_dataset.csv")
test_df = pd.read_csv("prediction_set.csv")

In [4]:
print(train_df.columns)
print(test_df.columns)

Index(['category_name', 'geography_name', 'wave_date', 'question_text',
       'qualifying_options', 'disqualifying_options', 'weight_3',
       'base_weight_3', 'percentage_3'],
      dtype='str')
Index(['prediction_id', 'category_name', 'geography_name', 'wave_date',
       'question_text', 'qualifying_options', 'disqualifying_options'],
      dtype='str')


Use `category_name` from both sets to generate the product classes and flags

In [10]:
product_categories = set(train_df.category_name).union(set(test_df.category_name))
print(f"{len(product_categories)} distinct product categories")

585 distinct product categories


In [14]:
with open("~/.openai-tracksuit-key") as f:
    tracksuit_key = f.readline()
    
client = OpenAI(api_key=tracksuit_key)
MODEL = "gpt-5-mini"

class Taxonomy(BaseModel):
    categories: list[str]

def generate_taxonomy(products):
    response = client.responses.parse(
        model=MODEL,
        instructions="""
        Design a general-purpose taxonomy for consumer
        product and service survey categories.

        Examine the entire product list before deciding.

        Create 10-20 broad, mutually exclusive categories.

        The categories must:
        - Cover all products in the supplied list.
        - Generalise to products not in the list.
        - Describe product domains, not behaviours.
        - Avoid categories specific to luxury or price.
        - Use short, consistent names.

        Return only the taxonomy.
        """,
        input=json.dumps(products),
        text_format=Taxonomy,
    )
    return response.output_parsed

taxonomy = generate_taxonomy(list(product_categories))
print(taxonomy.categories)

['Food & Grocery', 'Food & Beverage Services', 'Health & Medical', 'Beauty & Personal Care', 'Fashion & Accessories', 'Home & Furnishings', 'Home Services & Construction', 'Electronics & Technology', 'Sports, Fitness & Outdoors', 'Pets & Pet Care', 'Automotive & Mobility', 'Finance, Insurance & Payments', 'Real Estate & Property', 'Travel & Experiences', 'Media, Entertainment & Leisure', 'Professional & Business Services', 'Retail & Gifts']


In [20]:
with open("broad_categories.json","w") as f:
    json.dump(taxonomy.categories, f)

## Assign product categories to broad classes and extract product flags

In [24]:
# Produc category features

class ProductFeatures(BaseModel):
    product_category: str
    broad_category: str
    is_luxury: bool
    is_subscription: bool

    audience_breadth: int = Field(
        description=(
            "Estimate how broadly relevant this product or service "
            "category is to the general adult population, "
            "independently of the survey question or timeframe. "
            "Use a scale from 1 to 5: "
            "1 = very niche, relevant to a small minority; "
            "2 = specialised, relevant to a limited population; "
            "3 = moderately broad; "
            "4 = mainstream, relevant to most consumers; "
            "5 = nearly universal, relevant to almost everyone."
        ),
        ge=1,
        le=5,
    )

    purchase_frequency: Literal[
        "daily", "weekly", "monthly", "yearly",
        "every_few_years", "once_in_lifetime",
        "not_applicable"
    ] = Field(
        description=(
            "Expected natural purchase frequency for a typical "
            "consumer of this product, NOT the timeframe specified "
            "in the survey question. For services, use the typical "
            "frequency of purchasing or booking the service."
        )
    )

    usage_frequency: Literal[ "permanently", "several_times_a_day",
        "daily", "weekly", "monthly", "yearly",
        "every_few_years", "once_in_lifetime",
        "not_applicable"
    ]= Field(
        description=(
            "Estimate how often a typical "
            "consumer of this product would use it, NOT the timeframe specified "
            "in the survey question. For services, use the typical "
            "frequency of usage for example, Netflix daily, theatre subscription monthly..."
        )
    )

    
    cost_tier: int = Field(
        description=(
            "Estimate how the subjective cost of this product "
            "category for an average middle class buyer "
            "Use a scale from 1 to 5: "
            "1 = cheap, don't hesitate in buying it; "
            "2 = cheap enough to not think twice; "
            "3 = not too cheap but would buy it if needed; "
            "4 = expensive, think twice if it is needed; "
            "5 = very expensive, would have to really need it or want it."
        ),
        ge=1,
        le=5,
    )


class ProductBatch(BaseModel):
    items: list[ProductFeatures]

In [26]:
def classify_products(products, taxonomy):
    response = client.responses.parse(
        model=MODEL,
        instructions="""
        Classify consumer products and services.

        Use ONLY the supplied broad taxonomy categories.

        For each product, estimate its general characteristics:
        - is_luxury: whether the category explicitly represents
          luxury or premium products
        - is_subscription: whether recurring subscription is
          intrinsic to the category
        - product_type: the main form of product/service
        - purchase_frequency: typical natural purchase or usage
          frequency, independent of survey question wording
        - cost_tier: typical individual transaction cost, 1-5
        - audience_breadth: proportion of the general population
          for whom this category is relevant, 1-5

        Consider typical consumers rather than extreme cases.
        Preserve input product names exactly.
        Return exactly one item for each input product.
        """,
        input=json.dumps({
            "taxonomy": taxonomy.categories,
            "products": products,
        }),
        text_format=ProductBatch,
    )
    return response.output_parsed

In [30]:

batch_size = 30
results = []
products = list(product_categories)
for i in range(0, len(products), batch_size):
    batch = products[i:i + batch_size]

    result = classify_products(batch, taxonomy)

    items = [item.model_dump() for item in result.items]

    # Validate each batch
    #assert len(items) == len(batch)
    #assert {x["product_category"] for x in items} == set(batch)
    #assert all(
    #    x["broad_category"] in taxonomy.categories
    #    for x in items
    #)

    results.extend(items)

    # Save progress after every batch
    pd.DataFrame(results).to_json(
        "product_features.json",
        orient="records",
        indent=2,
    )

In [32]:
dfcat=pd.DataFrame(results)